# 01_agreement

Defines what should be built, who owns it, which rules apply, and what readiness means. This is the first required delivery notebook after `00_env_config`.

Required delivery flow: `01_agreement` → `02_pipeline` → `03_review`. Optional support lives in `99_explore`.

FabricOps v1 uses standalone widget cells because smaller widget outputs are more stable in Microsoft Fabric notebooks.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release  | Tested by | Date tested | 
|---|---|---| 
| v0.1.0 |  Voyce| 13 Jul 2026 | 


## 1. Run `00_env_config`


In [ ]:
%run 00_env_config


## 2. Import required functions


In [ ]:
from fabricops_kit import (
    widget_view_data_contract,
    # FabricOps v0.1.0 onwards
    widget_render_data_agreement,
    widget_render_data_steward,
)


## 3. Data steward

Run this standalone cell first to create or update data steward metadata. Smaller cells are more stable in Microsoft Fabric notebooks.


In [ ]:
steward_widget = widget_render_data_steward(spark=spark)


## 4. Data agreement

Run this standalone cell after at least one active steward exists.


In [ ]:
agreement_widget = widget_render_data_agreement(spark=spark)


## 5. View linked data contracts

Optional read-only visibility into contracts linked to the selected agreement. The agreement workflow remains valid when no contract is linked yet.


In [ ]:
agreement_contract_view = widget_view_data_contract(
    agreement=agreement_widget,
    spark_session=spark,
)


Select a dataset above, then rerun this cell to display its ten raw metadata traces.


In [ ]:
metadata_views = agreement_contract_view["get_views"]()

if metadata_views.get("error"):
    print(metadata_views["error"])
else:
    selection = metadata_views["selection"]
    print("Selected metadata scope")
    print(f"Environment: {selection.get('environment_name')}")
    print(f"Metadata table key: {selection.get('metadata_table_key')}")
    print(f"Agreement ID: {selection.get('agreement_id')}")

    filter_fields = {
        "METADATA_DATA_STEWARD": "steward_id",
        "METADATA_DATA_AGREEMENT": "agreement_id",
    }
    for table_name, frame in metadata_views["tables"].items():
        filter_field = filter_fields.get(table_name, "metadata_table_key")
        if table_name == "METADATA_DATA_STEWARD":
            filter_value = ", ".join(filter(None, (
                selection.get("provider_steward_id"),
                selection.get("recipient_steward_id"),
            )))
        else:
            filter_value = selection.get(filter_field)
        print(f"\n{table_name}")
        print(f"Filtered by {filter_field}: {filter_value}")
        print("Sorted by _committed_at descending")
        display(frame)
